In [1]:
# Imports 

import os
from PIL import Image
import cv2
import numpy as np
from gtda.homology import CubicalPersistence
from gtda.diagrams import PersistenceImage
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
path = '/Users/nathanielbrake/Downloads/archive-7'

In [8]:
def extract_features(image_path):
    img = np.array(Image.open(image_path).convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    raw = gray.astype(np.float32) / 255.0
    grad = gradient_magnitude(gray)
    features = []

    for data in [raw, grad]:
        diag = cp.fit_transform([data])
        pim = pi.fit_transform(diag)
        features.append(pim.flatten())

    return np.concatenate(features)


def gradient_magnitude(image):
    gx = cv2.Sobel(image, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(image, cv2.CV_32F, 0, 1)
    mag = np.sqrt(gx**2 + gy**2)
    return mag / (mag.max() + 1e-8)


def process_dataset_in_batches(data_dir, save_dir, batch_size=50):
    os.makedirs(save_dir, exist_ok=True)

    classes = {"REAL": 0, "FAKE": 1}

    X_batch = []
    y_batch = []
    batch_idx = 0

    for class_name, label in classes.items():
        class_path = os.path.join(data_dir, class_name)

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            try:
                feats = extract_features(img_path)
                X_batch.append(feats)
                y_batch.append(label)
            except Exception as e:
                print(f"Skipping {img_name}: {e}")
                continue

            if len(X_batch) >= batch_size:
                save_path = os.path.join(save_dir, f"batch_{batch_idx}.npz")

                np.savez(
                    save_path,
                    X=np.array(X_batch),
                    y=np.array(y_batch)
                )

                print(f"Saved {save_path}")

                X_batch, y_batch = [], []
                batch_idx += 1

    # Save remaining
    if X_batch:
        save_path = os.path.join(save_dir, f"batch_{batch_idx}.npz")
        np.savez_compressed(save_path, X=np.array(X_batch), y=np.array(y_batch))
        print(f"Saved final {save_path}")


In [9]:
cp = CubicalPersistence(homology_dimensions=[0, 1])
pi = PersistenceImage()

train_dir = path + "/train"
test_dir = path + "/test"

process_dataset_in_batches(
    train_dir,
    save_dir="train_batches",
    batch_size=500
)


✅ Saved train_batches/batch_0.npz
✅ Saved train_batches/batch_1.npz
✅ Saved train_batches/batch_2.npz
✅ Saved train_batches/batch_3.npz
✅ Saved train_batches/batch_4.npz
✅ Saved train_batches/batch_5.npz
✅ Saved train_batches/batch_6.npz
✅ Saved train_batches/batch_7.npz
✅ Saved train_batches/batch_8.npz
✅ Saved train_batches/batch_9.npz
✅ Saved train_batches/batch_10.npz
✅ Saved train_batches/batch_11.npz
✅ Saved train_batches/batch_12.npz
✅ Saved train_batches/batch_13.npz
✅ Saved train_batches/batch_14.npz
✅ Saved train_batches/batch_15.npz
✅ Saved train_batches/batch_16.npz
✅ Saved train_batches/batch_17.npz
✅ Saved train_batches/batch_18.npz
✅ Saved train_batches/batch_19.npz
✅ Saved train_batches/batch_20.npz
✅ Saved train_batches/batch_21.npz
✅ Saved train_batches/batch_22.npz
✅ Saved train_batches/batch_23.npz
✅ Saved train_batches/batch_24.npz
✅ Saved train_batches/batch_25.npz
✅ Saved train_batches/batch_26.npz
✅ Saved train_batches/batch_27.npz
✅ Saved train_batches/batch_28

In [10]:
process_dataset_in_batches(
    test_dir,
    save_dir="test_batches",
    batch_size=500
)

✅ Saved test_batches/batch_0.npz
✅ Saved test_batches/batch_1.npz
✅ Saved test_batches/batch_2.npz
✅ Saved test_batches/batch_3.npz
✅ Saved test_batches/batch_4.npz
✅ Saved test_batches/batch_5.npz
✅ Saved test_batches/batch_6.npz
✅ Saved test_batches/batch_7.npz
✅ Saved test_batches/batch_8.npz
✅ Saved test_batches/batch_9.npz
✅ Saved test_batches/batch_10.npz
✅ Saved test_batches/batch_11.npz
✅ Saved test_batches/batch_12.npz
✅ Saved test_batches/batch_13.npz
✅ Saved test_batches/batch_14.npz
✅ Saved test_batches/batch_15.npz
✅ Saved test_batches/batch_16.npz
✅ Saved test_batches/batch_17.npz
✅ Saved test_batches/batch_18.npz
✅ Saved test_batches/batch_19.npz
✅ Saved test_batches/batch_20.npz
✅ Saved test_batches/batch_21.npz
✅ Saved test_batches/batch_22.npz
✅ Saved test_batches/batch_23.npz
✅ Saved test_batches/batch_24.npz
✅ Saved test_batches/batch_25.npz
✅ Saved test_batches/batch_26.npz
✅ Saved test_batches/batch_27.npz
✅ Saved test_batches/batch_28.npz
✅ Saved test_batches/bat

In [2]:
path2 = '/Users/nathanielbrake/mycode/train_batches/'

def get_batches(i):
    batches = [f'batch_{i}.npz', f'batch_{i}.npz',f'batch_{i + 100}.npz', f'batch_{i+ 101}.npz'] 
    # Make sure you are getting batches with different labels
    X_list, y_list = [], []
    for file in batches:
        X, y = load_npz(path2 + file)
        #print(X)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)


def load_npz(file):
    data = np.load(file)
    X = data['X'] 
    y = data['y']
    return X.astype(np.float32), y




In [47]:
# Random Forest


# Obsolite, early try at training random forest, differ to multiple models methods below
rf = RandomForestClassifier(
    n_estimators=0,
    warm_start=True,
    n_jobs=1  # important for memory
)

trees_per_batch = 10


for i in range(0, 50): 
    X_batch, y_batch = get_batches(2*i)
    rf.n_estimators += trees_per_batch
    rf.fit(X_batch, y_batch)
    print(f'Trained batch number {i}')
    del X_batch, y_batch

Trained batch number 0
Trained batch number 1


KeyboardInterrupt: 

In [3]:
path3 = '/Users/nathanielbrake/mycode/test_batches'
def get_test_batches():
    X_list, y_list = [], []
    for img_name in os.listdir(path3):
        img_path = os.path.join(path3, img_name)
        X, y = load_npz(img_path)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)

In [40]:
X_test, y_test = get_test_batches()

y_pred = rf.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))




Accuracy: 0.67735

Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.50      0.61     10000
           1       0.63      0.85      0.73     10000

    accuracy                           0.68     20000
   macro avg       0.70      0.68      0.67     20000
weighted avg       0.70      0.68      0.67     20000



In [37]:
#del X_test, y_test

In [15]:
# Multiiple Model Random Forest
import numpy as np
import random
import gc
from collections import deque
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import random


In [16]:
def sample_params():
    return {
        "max_depth": random.choice([5, 10, 15]),
        "max_features": random.choice(["sqrt", "log2", 0.5, 0.8]),
        "min_samples_split": random.choice([5, 10, 15])
    }

# Stochasically randomize parameters
def mutate_params(params):
    new_params = params.copy()
    
    if random.random() < 0.5:
        new_params["max_depth"] = int(parent["params"]["max_depth"] * random.uniform(0.8, 1.2))
    
    if random.random() < 0.5:
        new_params["max_features"] = random.choice(["sqrt", "log2", 0.5, 0.8])
    
    if random.random() < 0.5:
        new_params["min_samples_split"] = int(parent['params']['min_samples_split'] * random.uniform(.5, 1.5))
    
    return new_params

In [17]:
def get_batches(i):
    batches = [f'batch_{i}.npz', f'batch_{i}.npz',f'batch_{i + 100}.npz', f'batch_{i+ 101}.npz'] 
    # Make sure you are getting batches with different labels
    X_list, y_list = [], []
    for file in batches:
        X, y = load_npz(path2 + file)
        #print(X)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)


# Function to evaluate models as they are running
def evaluate_model(model):
    y_true, y_pred = [], []
    X_batch, y_batch = get_test_batches_rand()
    #print(X_batch)
    preds = model.predict(X_batch)
    y_true.extend(y_batch)
    y_pred.extend(preds)
    a = accuracy_score(y_true, y_pred)
    return a

def get_test_batches_rand(n = 5):
    X_list, y_list = [], []
    for i in range(n):
        num = random.randint(0, 39)
        img_path = os.path.join(path3, f"batch_{num}.npz")
        X, y = load_npz(img_path)
        X_list.append(X)
        y_list.append(y)
    # print(X_list, y_list)
    return np.vstack(X_list), np.concatenate(y_list)

In [22]:

EVAL_INTERVAL = 5
trees_per_batch = 4
BUFFER_SIZE = 5
N_MODELS = 6
MIN_TREES_BEFORE_KILL = 20
KILL_THRESHOLD = 0.5  

buffer_X = deque(maxlen=BUFFER_SIZE)
buffer_y = deque(maxlen=BUFFER_SIZE)


models = []
for _ in range(N_MODELS):
    params = sample_params()
    rf = RandomForestClassifier(
        n_estimators=0,
        warm_start=True,
        n_jobs=1,
        **params
    )
    models.append({
        "model": rf,
        "params": params,
        "score": 0,
        "dead": False
    })
print(models)

[{'model': RandomForestClassifier(max_depth=10, max_features='log2', min_samples_split=10,
                       n_estimators=0, n_jobs=1, warm_start=True), 'params': {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 10}, 'score': 0, 'dead': False}, {'model': RandomForestClassifier(max_depth=5, max_features=0.8, min_samples_split=15,
                       n_estimators=0, n_jobs=1, warm_start=True), 'params': {'max_depth': 5, 'max_features': 0.8, 'min_samples_split': 15}, 'score': 0, 'dead': False}, {'model': RandomForestClassifier(max_depth=10, min_samples_split=10, n_estimators=0,
                       n_jobs=1, warm_start=True), 'params': {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 10}, 'score': 0, 'dead': False}, {'model': RandomForestClassifier(max_depth=10, max_features=0.8, min_samples_split=10,
                       n_estimators=0, n_jobs=1, warm_start=True), 'params': {'max_depth': 10, 'max_features': 0.8, 'min_samples_split': 10}, 'score': 0

In [24]:

def train_model(m, X, y):
    if m["dead"]:
        return m
    
    rf = m["model"]
    rf.n_estimators += trees_per_batch
    rf.fit(X, y)
    
    return m



for i in range(50):
    print(f"\nIteration {i}")
    
    # ---- load batch ----
    X_batch, y_batch = get_batches(2 * i)
    X_batch = X_batch.astype(np.float32)
    
    buffer_X.append(X_batch)
    buffer_y.append(y_batch)
    
    # ---- build training set ----
    X_train = np.vstack(buffer_X)
    y_train = np.concatenate(buffer_y)
    
    # ---- parallel training ----
    models = Parallel(n_jobs=-1)(
        delayed(train_model)(m, X_train, y_train)
        for m in models
    )
    
    # ---- evaluation ----
    if i % EVAL_INTERVAL == 0:
        print("Evaluating models...")
        
        for m in models:
            if not m["dead"]:
                m["score"] = evaluate_model(m["model"])
        
        # ---- sort models ----
        models.sort(key=lambda x: x["score"], reverse=True)
        
        print("Top score:", models[0]["score"])
        
        # ---- early pruning ----
        for m in models:
            if m["model"].n_estimators >= MIN_TREES_BEFORE_KILL:
                if m["score"] < KILL_THRESHOLD:
                    m["dead"] = True
        
        # ---- replace worst half ----
        num_replace = len(models) // 2
        
        for j in range(num_replace):
            parent = models[j]
            
            new_params = mutate_params(parent["params"])
            
            new_model = RandomForestClassifier(
                n_estimators=0,
                warm_start=True,
                n_jobs=1,
                **new_params
            )
            
            models[-(j+1)] = {
                "model": new_model,
                "params": new_params,
                "score": 0,
                "dead": False
            }
    
    # ---- cleanup ----
    del X_batch, y_batch, X_train, y_train
    gc.collect()


Iteration 0
Evaluating models...
Top score: 0.6716

Iteration 1

Iteration 2

Iteration 3

Iteration 4

Iteration 5
Evaluating models...
Top score: 0.75

Iteration 6

Iteration 7

Iteration 8

Iteration 9

Iteration 10
Evaluating models...
Top score: 0.7116

Iteration 11

Iteration 12

Iteration 13

Iteration 14

Iteration 15
Evaluating models...
Top score: 0.7132

Iteration 16

Iteration 17

Iteration 18

Iteration 19

Iteration 20
Evaluating models...
Top score: 0.7696

Iteration 21

Iteration 22

Iteration 23

Iteration 24

Iteration 25
Evaluating models...
Top score: 0.7392

Iteration 26

Iteration 27

Iteration 28

Iteration 29

Iteration 30
Evaluating models...
Top score: 0.7528

Iteration 31

Iteration 32

Iteration 33

Iteration 34

Iteration 35
Evaluating models...
Top score: 0.7584

Iteration 36

Iteration 37

Iteration 38

Iteration 39

Iteration 40
Evaluating models...
Top score: 0.7248

Iteration 41

Iteration 42

Iteration 43

Iteration 44

Iteration 45
Evaluating models

In [26]:
import joblib
import os

save_dir = "rf_models"
os.makedirs(save_dir, exist_ok=True)

for i, m in enumerate(models):
    model = m["model"]
    params = m["params"]
    score = m["score"]
    
    filename = f"{save_dir}/model_{i}_score_{score:.4f}.joblib"
    
    joblib.dump({
        "model": model,
        "params": params,
        "score": score
    }, filename)
    
    print(f"Saved: {filename}")

Saved: rf_models/model_0_score_0.7132.joblib
Saved: rf_models/model_1_score_0.7088.joblib
Saved: rf_models/model_2_score_0.7088.joblib
Saved: rf_models/model_3_score_0.0000.joblib
Saved: rf_models/model_4_score_0.0000.joblib
Saved: rf_models/model_5_score_0.0000.joblib


In [27]:
path3 = '/Users/nathanielbrake/mycode/test_batches'
def get_test_batches():
    X_list, y_list = [], []
    for img_name in os.listdir(path3):
        img_path = os.path.join(path3, img_name)
        X, y = load_npz(img_path)
        X_list.append(X)
        y_list.append(y)
    return np.vstack(X_list), np.concatenate(y_list)


X_test, y_test = get_test_batches()


for i, m in enumerate(models):
    model = m["model"]
    y_pred = model.predict(X_test)

    print("\nAccuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))



Accuracy: 0.70055

Classification Report:

              precision    recall  f1-score   support

           0       0.73      0.64      0.68     10000
           1       0.68      0.76      0.72     10000

    accuracy                           0.70     20000
   macro avg       0.70      0.70      0.70     20000
weighted avg       0.70      0.70      0.70     20000


Accuracy: 0.70425

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.66      0.69     10000
           1       0.69      0.75      0.72     10000

    accuracy                           0.70     20000
   macro avg       0.71      0.70      0.70     20000
weighted avg       0.71      0.70      0.70     20000


Accuracy: 0.6987

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.65      0.68     10000
           1       0.68      0.75      0.71     10000

    accuracy                           0.70     200